# cli

> Single `kosha` entry point with subcommands for shell-based harnesses.
> Default output is readable markdown; pass `--as-json` for JSON (piping/harnesses).

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, sys
from fastcore.script import call_parse, Param
from kosha.core import Kosha, env_pkg_versions

In [ ]:
#| export
def _to_json(obj):
    'Recursively convert kosha result objects to JSON-serializable types.'
    if isinstance(obj, (set, frozenset)): return sorted(obj)
    if isinstance(obj, dict): return {k: _to_json(v) for k, v in obj.items()}
    if hasattr(obj, '__iter__') and not isinstance(obj, str): return [_to_json(x) for x in obj]
    return obj

def _print_results(results):
    'Print context/search results as readable markdown blocks.'
    for r in results:
        m = r.get('metadata', {}) if isinstance(r, dict) else {}
        mod = m.get('mod_name', r.get('path', 'unknown')) if m else r.get('path', 'unknown')
        line = m.get('lineno', '?') if m else '?'
        pr = r.get('pagerank', 0) or 0
        print(f"\n## {mod}  (line {line})  [pagerank: {pr:.5f}]")
        content = (r.get('content', '') or '')[:300]
        if content: print(f"```\n{content}\n```")
        callers = r.get('callers', [])
        if callers: print(f"Callers: {', '.join(list(callers)[:5])}")

In [ ]:
# Test _to_json handles all expected types
from fastcore.foundation import L
assert _to_json({'a': {1, 2}}) == {'a': [1, 2]}
assert _to_json(L([{'x': 1}])) == [{'x': 1}]
assert _to_json('plain') == 'plain'
print("helpers ok")

In [ ]:
#| export
@call_parse
def sync(
    pkgs:str=None,       # comma-separated package names; defaults to all pyproject.toml deps
    parallel:bool=False, # run repo, env, and graph sync in parallel
    embed:bool=True,     # embed code chunks (set False for fast metadata-only update)
):
    'Sync repo + env packages + call graph into .kosha/ databases.'
    k = Kosha()
    pkg_list = pkgs.split(',') if pkgs else None
    k.sync(pkgs=pkg_list, in_parallel=parallel, embed=embed)

In [ ]:
#| export
@call_parse
def context(
    query:str,           # search query (supports key:value filters e.g. package:fastcore)
    limit:int=10,        # max results
    repo:bool=True,      # include repo results
    env:bool=True,       # include env results
    graph:bool=True,     # include call graph enrichment
    as_json:bool=False,  # output JSON instead of markdown
):
    'Fan-out semantic search over repo and env, optionally graph-enriched.'
    k = Kosha()
    results = k.context(query, limit=limit, repo=repo, env=env, graph=graph)
    if as_json: print(json.dumps(_to_json(results)))
    else: _print_results(results)

In [ ]:
# Test that functions are callable
assert callable(sync)
assert callable(context)
print("sync + context defined ok")

In [ ]:
#| export
@call_parse
def repo_context(
    query:str,          # search query
    limit:int=10,       # max results
    as_json:bool=False, # output JSON
):
    'Semantic + keyword search over indexed repo code only.'
    k = Kosha()
    results = k.repo_context(query, limit=limit)
    if as_json: print(json.dumps(_to_json(results)))
    else: _print_results(results)

In [ ]:
#| export
@call_parse
def env_context(
    query:str,          # search query (package names auto-detected as filters)
    limit:int=10,       # max results
    as_json:bool=False, # output JSON
):
    'Semantic search over indexed env packages only.'
    k = Kosha()
    results = k.env_context(query, limit=limit)
    if as_json: print(json.dumps(_to_json(results)))
    else: _print_results(results)

In [ ]:
#| export
@call_parse
def ni(
    mod_name:str,       # fully-qualified module node name e.g. fastcore.basics.merge
    as_json:bool=False, # output JSON
):
    'Node info: callers, callees, co_dispatched, pagerank for a single graph node.'
    k = Kosha()
    result = k.ni(mod_name)
    if as_json: print(json.dumps(_to_json(result)))
    else:
        print(f"\n## {mod_name}")
        print(f"pagerank: {result.get('pagerank', 0):.5f}")
        callers = result.get('callers', [])
        callees = result.get('callees', [])
        co = result.get('co_dispatched', [])
        if callers: print(f"Callers ({len(callers)}): {', '.join(list(callers)[:10])}")
        if callees: print(f"Callees ({len(callees)}): {', '.join(list(callees)[:10])}")
        if co: print(f"Co-dispatched: {', '.join(list(co)[:10])}")

In [ ]:
#| export
@call_parse
def watch(
    dir:str=None,  # directory to watch; defaults to repo root
):
    "Live incremental re-index on file changes. Blocking \u2014 Ctrl-C to stop."
    from pathlib import Path as _Path
    k = Kosha()
    k.watch_repo(dir=_Path(dir) if dir else None)

In [ ]:
#| export
@call_parse
def public_api(
    pkg:str,            # package name e.g. "fastcore" or submodule "fastcore.basics"
    module:str=None,    # restrict to a submodule (overrides pkg for scoping)
    as_json:bool=False, # output JSON
):
    "List public API entries for a package (respects __all__ + @patch methods)."
    k = Kosha()
    target = module or pkg
    results = k.public_api(target)
    if as_json: print(json.dumps(_to_json(results)))
    else:
        print(f"\n## Public API: {target}  ({len(results)} entries)")
        for r in results:
            name = r.get('mod_name', '')
            doc = r.get('docstring', '') or ''
            suffix = f'  # {doc.splitlines()[0][:60]}' if doc.strip() else ''
            print(f"  {name}{suffix}")

In [ ]:
#| export
@call_parse
def api_paths(
    from_pkg:str,       # source package (call origin)
    to_pkg:str,         # target package (call destination)
    k:int=15,           # top-k API nodes per package to consider
    as_json:bool=False, # output JSON
):
    "Shortest call-graph paths from from_pkg public API to to_pkg public API."
    ko = Kosha()
    paths = ko.api_call_paths(from_pkg, to_pkg, k=k)
    if as_json: print(json.dumps(_to_json(paths)))
    else:
        print(f"\n## Call paths: {from_pkg} \u2192 {to_pkg}  ({len(paths)} targets reached)")
        for target, path in sorted(paths.items(), key=lambda x: len(x[1])):
            print(f"\n  \u2192 {target}  (hops: {len(path)})")
            _arr = ' → '
        print(f"    {_arr.join(path)}")

In [ ]:
#| export
@call_parse
def dep_stack(
    seeds:str=None,     # comma-separated seed package names; defaults to pyproject.toml deps
    depth:int=1,        # BFS depth
    as_json:bool=False, # output JSON
):
    "BFS dependency layers from seed packages, ordered by coupling strength."
    k = Kosha()
    seed_list = seeds.split(',') if seeds else list(env_pkg_versions().keys())
    layers = k.dep_stack(seeds=seed_list, depth=depth)
    if as_json: print(json.dumps(_to_json(layers)))
    else:
        for i, layer in enumerate(layers):
            label = "seeds" if i == 0 else f"depth {i}"
            pkgs = sorted(layer) if isinstance(layer, (set, frozenset)) else layer
            print(f"\n## Layer {i} ({label}): {', '.join(pkgs)}")

In [ ]:
#| export
@call_parse
def top_nodes(
    pkg:str,            # package name e.g. "fastcore"
    k:int=5,            # number of top nodes to return
    as_json:bool=False, # output JSON
):
    "Top-k public API nodes for a package ranked by PageRank in the call graph."
    ko = Kosha()
    nodes = ko.top_nodes(pkg, k=k)
    if as_json: print(json.dumps(_to_json(nodes)))
    else:
        print(f"\n## Top {k} nodes: {pkg}")
        for node in nodes: print(f"  {node}")

In [ ]:
#| export
_DISPATCH = {
    'context':        lambda k, a: k.context(**a),
    'repo_context':   lambda k, a: k.repo_context(**a),
    'env_context':    lambda k, a: k.env_context(**a),
    'ni':             lambda k, a: k.ni(a['mod_name']),
    'graph_diff':     lambda k, a: k.graph_diff(),
    'find_similar':   lambda k, a: k.find_similar(**a),
    'surprising':     lambda k, a: k.surprising_connections(**a),
    'top_nodes':      lambda k, a: k.top_nodes(**a),
    'public_api':     lambda k, a: k.public_api(**a),
    'api_call_paths': lambda k, a: k.api_call_paths(**a),
    'sync':           lambda k, a: (k.sync(**a), 'synced')[1],
}

@call_parse
def daemon():
    "Persistent kosha kernel. Reads newline-delimited JSON from stdin, writes results to stdout."
    k = Kosha()
    print(json.dumps({"ok": True, "status": "ready"}), flush=True)
    for line in sys.stdin:
        if not (line := line.strip()): continue
        try:
            req = json.loads(line)
            fn = _DISPATCH[req['cmd']]
            print(json.dumps({"ok": True, "result": _to_json(fn(k, req.get('args', {})))}), flush=True)
        except Exception as e:
            print(json.dumps({"ok": False, "error": str(e)}), flush=True)


In [ ]:
#| hide
import tempfile as _td5
from pathlib import Path as _P5
from kosha.cli import _DISPATCH
with _td5.TemporaryDirectory() as _t:
    _p = _P5(_t); _k = Kosha(dir=_p, xdg_dir=_p)
    _k.gn.insert_all([
        {'node':'pkgA.foo','pagerank':0.10,'flavor':'function','file':'a.py'},
        {'node':'pkgB.bar','pagerank':0.20,'flavor':'function','file':'b.py'},
    ])
    _k.ge.insert({'caller':'pkgA.foo','callee':'pkgB.bar','kind':'static','confidence':1.0})

    # graph_diff — first call returns empty-delta dict
    _gd = _DISPATCH['graph_diff'](_k, {})
    assert set(_gd) == {'new_nodes','removed_nodes','new_edges','removed_edges','pagerank_shifts'}, _gd

    # top_nodes — accepts kwargs from request
    _tn = _DISPATCH['top_nodes'](_k, {'pkg':'pkgA','k':3})
    assert hasattr(_tn, '__iter__'), _tn

    # public_api — accepts kwargs
    _pa = _DISPATCH['public_api'](_k, {'pkg':'kosha'})
    assert hasattr(_pa, '__iter__'), _pa

    # surprising — uses use_embeddings flag
    _sp = _DISPATCH['surprising'](_k, {'top_n':5,'use_embeddings':False})
    assert hasattr(_sp, '__iter__'), _sp

    # ensure all expected keys are wired
    assert set(_DISPATCH) >= {'context','repo_context','env_context','ni','graph_diff',
                              'find_similar','surprising','top_nodes','public_api',
                              'api_call_paths','sync'}
print('_DISPATCH smoke test ok')

In [ ]:
#| hide
#| eval: false
import json as _json, subprocess as _sp, sys as _sys, tempfile as _td6
from pathlib import Path as _P6
with _td6.TemporaryDirectory() as _t:
    _p = _P6(_t)
    # spawn daemon; force Kosha() to use _t by chdir into it and pointing XDG_DATA_HOME
    _env = dict(**__import__('os').environ, XDG_DATA_HOME=str(_p))
    _proc = _sp.Popen([_sys.executable, '-c', 'from kosha.cli import daemon; daemon()'],
                      cwd=str(_p),
                      stdin=_sp.PIPE, stdout=_sp.PIPE, stderr=_sp.PIPE,
                      text=True, env=_env)
    # ready line
    _ready = _json.loads(_proc.stdout.readline())
    assert _ready == {'ok': True, 'status': 'ready'}, _ready
    # graph_diff via daemon
    _proc.stdin.write(_json.dumps({'cmd':'graph_diff'}) + '\n'); _proc.stdin.flush()
    _resp = _json.loads(_proc.stdout.readline())
    assert _resp['ok'] is True and isinstance(_resp['result'], dict), _resp
    # malformed line yields ok:false
    _proc.stdin.write('not json\n'); _proc.stdin.flush()
    _err = _json.loads(_proc.stdout.readline())
    assert _err['ok'] is False and 'error' in _err, _err
    _proc.stdin.close(); _proc.wait(timeout=5)
print('daemon e2e ok')

In [ ]:
#| export
@call_parse
def diff(as_json:bool=False):
    "Show delta between current graph and last snapshot (run after kosha sync)."
    k = Kosha()
    d = k.graph_diff()
    if as_json: print(json.dumps(_to_json(d)))
    else:
        print(f"New nodes ({len(d['new_nodes'])}): {', '.join(d['new_nodes'][:10])}")
        print(f"Removed  ({len(d['removed_nodes'])}): {', '.join(d['removed_nodes'][:10])}")
        print(f"New edges: {len(d['new_edges'])}  Removed edges: {len(d['removed_edges'])}")
        _arr = ' → '
        for node, old, new in d['pagerank_shifts'][:5]:
            print(f"  PageRank shift: {node}  {old:.5f}{_arr}{new:.5f}")


In [ ]:
#| hide
import io as _io, json as _json, contextlib as _cl, tempfile as _td7
from pathlib import Path as _P7
from kosha.cli import diff as _diff_cli
with _td7.TemporaryDirectory() as _t:
    _p = _P7(_t); _k = Kosha(dir=_p, xdg_dir=_p)
    _k.gn.insert({'node':'A','pagerank':0.1,'flavor':'function','file':'a.py'})
    # diff prints JSON; capture and parse. Patch Kosha() inside diff to use ours.
    import kosha.cli as _cli
    _orig = _cli.Kosha
    _cli.Kosha = lambda: _k
    try:
        _buf = _io.StringIO()
        with _cl.redirect_stdout(_buf): _diff_cli(as_json=True)
        _out = _json.loads(_buf.getvalue())
        assert set(_out) == {'new_nodes','removed_nodes','new_edges','removed_edges','pagerank_shifts'}, _out
    finally:
        _cli.Kosha = _orig
print('diff CLI ok')

In [ ]:
#| export
@call_parse
def find_similar_cli(
    node:str,           # fully-qualified node e.g. fastcore.basics.merge
    k:int=10,           # number of similar nodes
    as_json:bool=False, # output JSON
):
    "Find k most embedding-similar nodes (semantic peers, not call-graph neighbors)."
    ko = Kosha()
    results = ko.find_similar(node, k=k)
    if as_json: print(json.dumps(_to_json(results)))
    else: _print_results(results)

In [ ]:
#| hide
import io as _io2, json as _json2, contextlib as _cl2, numpy as _np2, tempfile as _td8
from pathlib import Path as _P8
from kosha.cli import find_similar_cli as _fs_cli
with _td8.TemporaryDirectory() as _t:
    _p = _P8(_t); _k = Kosha(dir=_p, xdg_dir=_p)
    _e = lambda v: _np2.array(v, dtype=_np2.float16).tobytes()
    _k.code_st.insert(dict(path='a.py',  content='A',  embedding=_e([1.0,0.0,0.0]), metadata='{\"mod_name\":\"A\"}'))
    _k.code_st.insert(dict(path='a2.py', content='A2', embedding=_e([0.99,0.01,0.0]), metadata='{\"mod_name\":\"A2\"}'))
    import kosha.cli as _cli
    _orig = _cli.Kosha; _cli.Kosha = lambda: _k
    try:
        _buf = _io2.StringIO()
        with _cl2.redirect_stdout(_buf): _fs_cli(node='A', k=2, as_json=True)
        _out = _json2.loads(_buf.getvalue())
        assert isinstance(_out, list), _out
        _names = [r.get('metadata',{}).get('mod_name') for r in _out]
        assert 'A' not in _names and 'A2' in _names, _names
    finally:
        _cli.Kosha = _orig
print('find_similar CLI ok')

In [ ]:
#| export
@call_parse
def surprising(
    top_n:int=10,             # number of surprising connections to show
    no_embeddings:bool=False, # skip embedding distance scoring (faster)
    as_json:bool=False,       # output JSON
):
    "Show top surprising cross-module call relationships by structural + semantic surprise score."
    k = Kosha()
    results = k.surprising_connections(top_n=top_n, use_embeddings=not no_embeddings)
    if as_json: print(json.dumps(_to_json(results)))
    else:
        _arr = ' → '
        for s in results:
            emb = f" emb_dist={s['embedding_distance']}" if s.get('embedding_distance') is not None else ''
            print(f"{s['caller']}{_arr}{s['callee']}  kind={s['kind']} surprise={s['surprise_score']}{emb}")


In [ ]:
#| hide
import io as _io3, json as _json3, contextlib as _cl3, tempfile as _td9
from pathlib import Path as _P9
from kosha.cli import surprising as _sur_cli
with _td9.TemporaryDirectory() as _t:
    _p = _P9(_t); _k = Kosha(dir=_p, xdg_dir=_p)
    _k.gn.insert_all([
        {'node':'pkgA.foo','pagerank':0.1,'flavor':'function','file':'a.py'},
        {'node':'pkgB.bar','pagerank':0.2,'flavor':'function','file':'b.py'},
    ])
    _k.ge.insert({'caller':'pkgA.foo','callee':'pkgB.bar','kind':'static','confidence':1.0})
    import kosha.cli as _cli
    _orig = _cli.Kosha; _cli.Kosha = lambda: _k
    try:
        _buf = _io3.StringIO()
        with _cl3.redirect_stdout(_buf): _sur_cli.__wrapped__(top_n=5, no_embeddings=True, as_json=True)
        _out = _json3.loads(_buf.getvalue())
        assert isinstance(_out, list) and len(_out) >= 1, _out
        assert any(s['cross_module'] for s in _out), _out
    finally:
        _cli.Kosha = _orig
print('surprising CLI ok')

In [ ]:
#| export
CMDS = {
    'sync':           sync,
    'context':        context,
    'repo-context':   repo_context,
    'env-context':    env_context,
    'ni':             ni,
    'watch':          watch,
    'public-api':     public_api,
    'api-paths':      api_paths,
    'dep-stack':      dep_stack,
    'top-nodes':      top_nodes,
    'daemon':         daemon,
    'diff':           diff,
    'find-similar':   find_similar_cli,
    'surprising':     surprising,
}

def main():
    "Entry point for the `kosha` CLI command."
    if len(sys.argv) < 2 or sys.argv[1] not in CMDS:
        cmds = ' | '.join(CMDS)
        print(f"Usage: kosha [{cmds}]")
        sys.exit(0 if len(sys.argv) < 2 else 1)
    cmd = sys.argv.pop(1)
    CMDS[cmd]()


In [ ]:
# Test dispatcher has all expected commands
assert set(CMDS.keys()) == {
    'sync', 'context', 'repo-context', 'env-context', 'ni',
    'watch', 'public-api', 'api-paths', 'dep-stack', 'top-nodes',
    'daemon', 'diff', 'find-similar', 'surprising'
}
print(f"CMDS registered: {sorted(CMDS)}")
